---
<img style="float: right; margin: 15px 15px 15px 15px;" src="https://images.seeklogo.com/logo-png/51/2/reddit-logo-png_seeklogo-511297.png" width="350px" height="120px" />

# <font color=#bbc28d>**AITA — Moral Judgment LLM**</font>
#### <font color=#2E9AFE>`Dataset Preparation Pipeline`</font>

---

## <font color= #66b0b0> &ensp; • **Interactive Demo** </font>

We wrap the fine-tuned model in a Gradio interface for interactive inference. The `judge` function applies the same chat template and generation pipeline used during evaluation — with an optional temperature parameter that switches between greedy decoding at 0.0 and nucleus sampling at higher values. The UI exposes a free-text input, generation controls hidden behind an accordion, and an HTML verdict panel that renders the predicted class as a styled badge alongside the model's reasoning with the raw verdict token stripped from the body text.

`NOTE: We ran this pipeline on a A100 GPU, however possible loading can be done on a T4, you just need to adapt the model loading section to use bitsandbytes and float 16.`

In [ ]:
# Install dependencies
!pip install -q transformers peft bitsandbytes datasets accelerate trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 44.9 MB/s eta 0:00:00


In [ ]:
import gradio as gr
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

In [ ]:
# Global config
BASE_MODEL   = "Qwen/Qwen2.5-7B-Instruct"
ADAPTER_REPO = "pipo1313/aita-lora-adapter" # Fintunned tunned layer

In [ ]:
# Model propmpt base
SYSTEM_PROMPT = (
    "You are an impartial moral judge. When the user describes an interpersonal "
    "conflict, issue a verdict using exactly one of these four codes:\n\n"
    "• NTA — Not The Asshole: the user is not at fault\n"
    "• YTA — You're The Asshole: the user acted wrongly\n"
    "• ESH — Everyone Sucks Here: all parties share some blame\n"
    "• NAH — No Assholes Here: no one acted badly, it's just a disagreement\n\n"
    "Always start your response with the verdict code, then explain your reasoning "
    "in 3–5 sentences. Be direct and honest. Do not hedge or over-qualify."
)

# Veredicts
VERDICT_STYLES = {
    "NTA": {"color": "#6e7f5b", "bg": "#d4f5e2", "label": "Not The Asshole"},
    "YTA": {"color": "#582121", "bg": "#fde8e8", "label": "You're The Asshole"},
    "ESH": {"color": "#4c3b5c", "bg": "#fef3c7", "label": "Everyone Sucks Here"},
    "NAH": {"color": "#547a7c", "bg": "#e1effe", "label": "No Assholes Here"},
}

EXAMPLES = [
    "I refused to lend money to my brother who has never paid me back. AITA?",
    "I told my coworker her presentation was bad in front of the whole team. AITA?",
    "I skipped my best friend's wedding because I wasn't feeling well. AITA?",
    "I didn't invite my half-siblings to my birthday dinner because we barely know each other. AITA?",
    "I told my mom I wouldn't be coming home for Christmas this year. AITA?",
]

In [ ]:
print("Loading model...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(base_model, "pipo1313/aita-lora-adapter", device_map="auto")
model.eval()

print("Model ready.")

Loading model...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Model ready.


In [ ]:
# Run Inference
def judge(situation, temperature, max_tokens):
    situation = situation.strip()
    if not situation:
        return "<p style='color:#888'>Please describe a situation first.</p>"

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": situation},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    use_sampling = temperature > 0.0

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=int(max_tokens),
            do_sample=use_sampling,
            temperature=float(temperature) if use_sampling else None,
            top_p=0.9 if use_sampling else None,
            repetition_penalty=1.3,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Detect verdict and strip it from the body text
    import re
    match = re.search(r'\b(NTA|YTA|ESH|NAH)\b', text.upper())
    verdict_code = match.group(1) if match else None
    style = VERDICT_STYLES.get(verdict_code)

    if style:
        # Strip everything up to and including the verdict code
        body = re.sub(r'(?i)^.*?(NTA|YTA|ESH|NAH)[:\s—-]*', '', text, count=1).strip()
        badge_html = f"""
        <div style="
            display: inline-block;
            padding: 6px 18px;
            border-radius: 999px;
            background: {style['bg']};
            color: {style['color']};
            font-size: 1.1em;
            font-weight: 700;
            letter-spacing: 0.05em;
            margin-bottom: 14px;
        ">{verdict_code} — {style['label']}</div>
        <p style="
            font-size: 1em;
            line-height: 1.75;
            color: var(--body-text-color);
            margin: 0;
        ">{body}</p>
        """
    else:
        badge_html = f"<p style='line-height:1.75'>{text}</p>"

    return badge_html


In [ ]:
# UI
CSS = """
#title { text-align: center; }
#subtitle { text-align: center; color: #555; margin-top: -10px; }
#disclaimer { text-align: center; font-size: 0.8em; color: #888; margin-top: 6px; }
footer { display: none !important; }
body, .gradio-container { background-color: #9ea87e !important; }
.block, .panel { background-color: #ffffff !important; }
button.primary { background-color: #ffffff !important; border-color: #9ea87e !important; color: #020300 !important; }
input[type=range].svelte-1kajgn1 {
    accent-color: #9ea87e;
    --track-color: #9ea87e;
}
input[type=range].svelte-1kajgn1::-webkit-slider-runnable-track {
    background: linear-gradient(to right, #9ea87e calc(var(--range_progress)), #ddd calc(var(--range_progress))) !important;
}
input[type=range].svelte-1kajgn1::-webkit-slider-thumb {
    background: #9ea87e !important;
}
"""

with gr.Blocks(css=CSS) as demo:

    gr.Markdown("# AITA — Moral Judgment", elem_id="title")
    gr.Markdown("Describe your situation and get a verdict.", elem_id="subtitle")

    with gr.Row():

        # Input
        with gr.Column(scale=3):
            situation_box = gr.Textbox(
                label="What happened?",
                placeholder="Describe your situation in first person...",
                lines=7,
                max_lines=12,
            )

            with gr.Accordion("⚙️  Generation settings", open=False):
              temperature = gr.Slider(
                  minimum=0.0, maximum=1.0, value=0.0, step=0.05,
                  label="Temperature (0 = deterministic, higher = more creative)",
              )
              max_tokens = gr.Slider(
                  minimum=60, maximum=350, value=150, step=10,
                  label="Max response length (tokens)",
              )

            submit_btn = gr.Button("Judge", variant="primary", size="lg")

        # Output
        with gr.Column(scale=3):
            verdict_box = gr.HTML(
                value="<p style='color:#aaa'>Your verdict will appear here.</p>",
                label="Verdict",
            )

    gr.Markdown(
        "*Research prototype — verdicts reflect Reddit community patterns, "
        "not universal ethics. Do not use for real decisions.*",
        elem_id="disclaimer",
    )

    gr.Examples(
        examples=EXAMPLES,
        inputs=situation_box,
        label="Try an example",
        examples_per_page=5,
    )

    # Show/hide temperature slider based on sampling toggle
    submit_btn.click(
        fn=judge,
        inputs=[situation_box, temperature, max_tokens],
        outputs=verdict_box,
    )
    situation_box.submit(
        fn=judge,
        inputs=[situation_box, temperature, max_tokens],
        outputs=verdict_box,
    )

demo.launch(share=True, debug=True)

/tmp/ipykernel_2889/3451863517.py:21: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CSS) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://07eb28040ff5fbaf75.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7861 <> https://07eb28040ff5fbaf75.gradio.live
